In [20]:
import os
from kaggle_secrets import UserSecretsClient
from pathlib import Path

user_secrets = UserSecretsClient()

BASE_URL = "https://neglegere.com"
EMAIL = user_secrets.get_secret("email")
PASSWORD = user_secrets.get_secret("password")
SAVE_DIR = Path("/kaggle/working/reference_audio")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_TOKEN"] = user_secrets.get_secret("hf_token")

R2_ACCOUNT_ID = user_secrets.get_secret("r2_AccountID")
R2_ACCESS_KEY = user_secrets.get_secret("r2_accesskey")
R2_SECRET_KEY = user_secrets.get_secret("r2_secretkey")
R2_BUCKET_NAME = user_secrets.get_secret("r2_bucket_name")
R2_ENDPOINT = f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com"

WEBHOOK_URL = user_secrets.get_secret("webhook_url")


def notify_slack(text: str):
    import requests
    try:
        requests.post(WEBHOOK_URL, json={"text": text}, timeout=15)
    except Exception as e:
        print("Slack通知に失敗:", e)

In [15]:
!pip install torch soundfile transformers accelerate
!pip install qwen-tts librosa scipy

import os
import torch
import soundfile as sf
from pathlib import Path
from qwen_tts import Qwen3TTSModel


def generate_qwen_voice(
    text: str,
    ref_audio_path: str,
    ref_text: str,
    output_path: str = "/kaggle/working/qwen_generated.wav",
    model_name: str = "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
):
    """
    Qwen3-TTSを使用してボイスクローン音声を生成するメソッド。

    Args:
        text (str): 生成する音声のテキスト
        ref_audio_path (str): 参照音声（WAVファイル）のパス
        ref_text (str): 参照音声の読み上げ内容
        output_path (str): 生成された音声の保存先
        model_name (str): 使用するQwenモデル

    Returns:
        str: 生成された音声ファイルのパス
    """

    # デバイス判定
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # モデルの読み込み
    model = Qwen3TTSModel.from_pretrained(
        model_name,
        device_map="auto" if device == "cuda" else "cpu",
        dtype=torch.float16 if device == "cuda" else torch.float32,
    )

    print(f"✅ Model loaded on {device}")

    # 音声生成
    wavs, sr = model.generate_voice_clone(
        text=text,
        language="Japanese",
        ref_audio=ref_audio_path,
        ref_text=ref_text,
    )

    # 保存先のディレクトリを作成
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # WAVファイルとして保存
    sf.write(str(output_path), wavs[0], sr)

    print(f"✅ Generated voice saved: {output_path}")

    return str(output_path)

In [16]:
import requests
from pathlib import Path

def login() -> str:
    url = f"{BASE_URL}/api/auth/login"
    headers = {
        "accept": "application/json",
        "Content-Type": "application/json",
    }
    payload = {
        "email": EMAIL,
        "password": PASSWORD,
    }

    res = requests.post(url, headers=headers, json=payload, timeout=30)
    res.raise_for_status()

    data = res.json()

    token = data.get("token")

    return token

def get_pending_items(token: str) -> list[dict]:
    url = f"{BASE_URL}/api/voice_clones/pending"
    headers = {
        "accept": "application/json",
        "Authorization": f"Bearer {token}",
    }

    res = requests.get(url, headers=headers, timeout=30)
    res.raise_for_status()

    data = res.json()

    items = data.get("items", [])

    return items

def download_reference_audio(token: str, voice_clone_id: str, file_name: str | None = None) -> Path:
    url = f"{BASE_URL}/api/reference_audio/{voice_clone_id}/file"
    headers = {
        "accept": "audio/wav",
        "Authorization": f"Bearer {token}",
    }

    res = requests.post(url, headers=headers, data="", stream=True, timeout=60)
    res.raise_for_status()

    output_name = Path(file_name).name

    output_path = SAVE_DIR / output_name

    with open(output_path, "wb") as f:
        for chunk in res.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

    return output_path

def upload_completed_voice_clone(
    token: str,
    voice_clone_id: str,
    audio_file_path,
):
    audio_file_path = Path(audio_file_path)

    url = f"{BASE_URL}/api/voice_clones/{voice_clone_id}/complete"

    headers = {
        "accept": "application/json",
        "Authorization": f"Bearer {token}",
    }

    with open(audio_file_path, "rb") as f:
        files = {
            "file": (audio_file_path.name, f, "audio/wav")
        }

        res = requests.post(
            url,
            headers=headers,
            files=files,
            timeout=60,
        )

    res.raise_for_status()

    data = res.json()

    if not data.get("ok"):
        raise Exception(f"APIエラー: {data}")

    return data

In [18]:
import boto3
import json

def fetch_pending_voice_clones(key: str = "pending.json"):
    s3 = boto3.client(
        "s3",
        endpoint_url=R2_ENDPOINT,
        aws_access_key_id=R2_ACCESS_KEY,
        aws_secret_access_key=R2_SECRET_KEY,
    )

    res = s3.get_object(
        Bucket=R2_BUCKET_NAME,
        Key=key,
    )

    return json.loads(res["Body"].read().decode("utf-8"))

print(fetch_pending_voice_clones())

{'items': [{'id': 'acff0201-8c24-4872-8c6a-66223b599626', 'referenceAudioPath': '7eb76c40-c81b-440c-a762-757ccf0c69e8.wav', 'recordedText': 'あの、最近少し暖かくなってきましたね。えっと、季節の変わり目は体調を崩しやすいので、無理をせずに過ごすことが大切だと思います。そうですね、しっかり休んで、毎日を元気に過ごしていきたいですね。', 'desiredText': 'テストです'}]}


In [ ]:
import shutil
import time

def main():

    i = 0

    while True:
        print(f"{i} 回目")
        i += 1
        time.sleep(10)
        items = fetch_pending_voice_clones()
        
        if not items:
            print("pending は 0 件です")
            continue
    
        for item in items["items"]:
            Path("/kaggle/working/reference_audio").mkdir(parents=True, exist_ok=True)
            Path("/kaggle/working/clone").mkdir(parents=True, exist_ok=True)
        
            voice_clone_id = item["id"]
            referenceAudioPath = item["referenceAudioPath"]
    
            print(f"downloading id={voice_clone_id} ...")
            token = login()
            output_path = download_reference_audio(
                token, 
                voice_clone_id, 
                referenceAudioPath
            )
            print(f"saved: {output_path}")
    
            # 音声生成
            output_file = generate_qwen_voice(
                text=item["desiredText"],
                ref_audio_path=str(output_path),
                ref_text=item["recordedText"],
                output_path=Path("/kaggle/working/clone/") / Path(referenceAudioPath).name,
            )
    
            upload_completed_voice_clone(
                token,
                voice_clone_id,
                output_file
            )
            
            print("生成されたファイル:", output_file)
            
            shutil.rmtree("/kaggle/working/reference_audio")
            shutil.rmtree("/kaggle/working/clone")

    print("\n完了")

import traceback

try:
    main()
except Exception:
    tb = traceback.format_exc()
    print(tb)
    notify_slack(
        f":rotating_light: voice-clone-task が失敗しました\n```\n{tb[-3000:]}\n```"
    )
    raise


0 回目
downloading id=acff0201-8c24-4872-8c6a-66223b599626 ...
saved: /kaggle/working/reference_audio/7eb76c40-c81b-440c-a762-757ccf0c69e8.wav


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

✅ Generated voice saved: /kaggle/working/clone/88ec0ba8-dc24-404e-a402-e82e2ead8d9c.wav
生成されたファイル: /kaggle/working/clone/88ec0ba8-dc24-404e-a402-e82e2ead8d9c.wav
41 回目
42 回目
43 回目
44 回目
45 回目
46 回目
47 回目
48 回目
49 回目
50 回目
51 回目
52 回目
53 回目
54 回目
55 回目
56 回目
57 回目
58 回目
59 回目
60 回目
61 回目
62 回目
63 回目
64 回目
65 回目
66 回目
67 回目
68 回目
69 回目
70 回目
71 回目
72 回目
73 回目
74 回目
75 回目
76 回目
77 回目
78 回目
79 回目
80 回目
81 回目
82 回目
83 回目
84 回目
85 回目
86 回目
87 回目
88 回目
89 回目
90 回目
91 回目
92 回目
93 回目
94 回目
95 回目
96 回目
97 回目
98 回目
99 回目
100 回目
101 回目
102 回目
103 回目
104 回目
105 回目
downloading id=300b9bd1-2809-4a35-8f79-03fbe46fb070 ...
saved: /kaggle/working/reference_audio/600c06f8-19cb-4ca8-a4f3-467b61f1e50a.wav


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Model loaded on cpu


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


✅ Generated voice saved: /kaggle/working/clone/600c06f8-19cb-4ca8-a4f3-467b61f1e50a.wav
生成されたファイル: /kaggle/working/clone/600c06f8-19cb-4ca8-a4f3-467b61f1e50a.wav
106 回目
107 回目
108 回目
109 回目
110 回目
111 回目
112 回目
113 回目
114 回目
115 回目
116 回目
117 回目
118 回目
119 回目
120 回目
121 回目
122 回目
123 回目
124 回目
125 回目
126 回目
127 回目
128 回目
129 回目
130 回目
131 回目
132 回目
133 回目
134 回目
143 回目
144 回目
145 回目
146 回目
147 回目
148 回目
149 回目
150 回目
151 回目
152 回目
153 回目
154 回目
155 回目
156 回目
157 回目
158 回目
159 回目
160 回目
161 回目
162 回目
163 回目
164 回目
165 回目
166 回目
167 回目
168 回目
169 回目
170 回目
171 回目
172 回目
173 回目
174 回目
175 回目
176 回目
177 回目
178 回目
179 回目
180 回目
181 回目
182 回目
183 回目
184 回目
185 回目
186 回目
187 回目
188 回目
189 回目
190 回目
191 回目
192 回目
193 回目
194 回目
195 回目
196 回目
197 回目
198 回目
199 回目
200 回目
201 回目
202 回目
203 回目
204 回目
205 回目
206 回目
207 回目
208 回目
209 回目
210 回目
211 回目
212 回目
213 回目
214 回目
215 回目
216 回目
217 回目
218 回目
219 回目
220 回目
221 回目
222 回目
223 回目
224 回目
225 回目
226 回目
227 回目
228 回目
229 回目
230 回目
231 回目
232 回目
233 回